In [ ]:
# Section 6: Initialize YOLO Model and Test on Sample Video

print("Initializing YOLO detector...")
detector = YOLODetector(model_name='yolov8n.pt', conf_threshold=0.5)

if detector.load():
    print("\n" + "="*60)
    print("YOLO Model Ready!")
    print("="*60)
else:
    print("⚠️  Model loading failed. Install ultralytics: pip install ultralytics")

# Show model info
if detector.model:
    print(f"\nModel Info:")
    print(f"  - Type: {detector.model.type}")
    print(f"  - Task: {detector.model.task}")
    print(f"  - Size: {detector.model.model_name if hasattr(detector.model, 'model_name') else 'nano'}")

In [ ]:
# Section 5: Create YOLO Model Wrapper for Object Detection

class YOLODetector:
    """Wrapper class for YOLOv8 object detection on video frames"""
    
    def __init__(self, model_name='yolov8n.pt', conf_threshold=0.5, iou_threshold=0.45):
        """
        Initialize YOLO detector
        
        Args:
            model_name: Name of YOLO model ('yolov8n.pt', 'yolov8s.pt', etc.)
            conf_threshold: Confidence threshold for detections
            iou_threshold: IOU threshold for NMS
        """
        self.model_name = model_name
        self.conf = conf_threshold
        self.iou = iou_threshold
        self.model = None
        
    def load(self):
        """Load the YOLO model"""
        try:
            print(f"Loading {self.model_name}...")
            from ultralytics import YOLO
            self.model = YOLO(self.model_name)
            print(f"✅ Model loaded successfully")
            return True
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            return False
    
    def predict(self, frame, return_boxes=True, visualize=True):
        """
        Run inference on a frame
        
        Args:
            frame: Input frame (RGB format)
            return_boxes: Whether to return raw boxes
            visualize: Whether to return annotated image
        
        Returns:
            Dictionary with detection results
        """
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load() first.")
        
        # Run inference
        results = self.model(frame, conf=self.conf, iou=self.iou, verbose=False)
        
        # Extract results
        boxes = results[0].boxes
        class_ids = boxes.cls.cpu().numpy().astype(int)
        confidences = boxes.conf.cpu().numpy()
        bboxes = boxes.xyxy.cpu().numpy().astype(int)
        
        # Filter for persons only (class 0)
        person_mask = class_ids == 0
        person_count = np.sum(person_mask)
        person_boxes = bboxes[person_mask]
        person_confs = confidences[person_mask]
        
        result_dict = {
            'count': person_count,
            'boxes': person_boxes,
            'confidences': person_confs,
            'all_boxes': bboxes,
            'all_classes': class_ids,
            'all_confs': confidences
        }
        
        # Generate annotated image
        if visualize:
            annotated = frame.copy()
            
            # Draw detected persons
            for box, conf in zip(person_boxes, person_confs):
                x1, y1, x2, y2 = box
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(annotated, f'Person: {conf:.2f}', 
                           (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                           0.5, (0, 255, 0), 2)
            
            # Draw count
            cv2.putText(annotated, f'Count: {person_count}', 
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 
                       1, (0, 255, 0), 2)
            
            result_dict['annotated_image'] = annotated
        
        return result_dict

print("✅ YOLODetector class defined")

In [ ]:
# Section 4: Define Object Detection and Visualization Functions

# COCO class names (80 classes in YOLOv8)
COCO_CLASSES = {
    0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane',
    5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light',
    10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench',
    14: 'cat', 15: 'dog', 16: 'horse', 17: 'sheep', 18: 'cow', 19: 'elephant',
    20: 'bear', 21: 'zebra', 22: 'giraffe', 23: 'backpack', 24: 'umbrella',
    25: 'handbag', 26: 'tie', 27: 'suitcase', 28: 'frisbee', 29: 'skis',
    30: 'snowboard', 31: 'sports ball', 32: 'kite', 33: 'baseball bat',
    34: 'baseball glove', 35: 'skateboard', 36: 'surfboard', 37: 'tennis racket',
    38: 'bottle', 39: 'wine glass', 40: 'cup', 41: 'fork', 42: 'knife',
    43: 'spoon', 44: 'bowl', 45: 'banana', 46: 'apple', 47: 'sandwich',
    48: 'orange', 49: 'broccoli', 50: 'carrot', 51: 'hot dog', 52: 'pizza',
    53: 'donut', 54: 'cake', 55: 'chair', 56: 'couch', 57: 'potted plant',
    58: 'bed', 59: 'dining table', 60: 'toilet', 61: 'tv', 62: 'laptop',
    63: 'mouse', 64: 'remote', 65: 'keyboard', 66: 'microwave', 67: 'oven',
    68: 'toaster', 69: 'sink', 70: 'refrigerator', 71: 'book', 72: 'clock',
    73: 'vase', 74: 'scissors', 75: 'teddy bear', 76: 'hair drier', 77: 'toothbrush',
    78: 'person_large', 79: 'person_small'
}

def process_video_frames(video_path, model, max_frames=None, sample_every=1, show_every=10):
    """
    Process video frames and detect objects
    
    Args:
        video_path: Path to video file
        model: YOLO model object
        max_frames: Maximum frames to process (None = all)
        sample_every: Process every Nth frame (for speed)
        show_every: Display every Nth result frame
    
    Returns:
        Dictionary with statistics and sample frames
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Could not open video: {video_path}")
        return None
    
    results = {
        'frames_processed': 0,
        'detections': defaultdict(int),
        'person_counts': [],
        'frame_times': [],
        'sample_frames': []
    }
    
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # Skip frames if sample_every > 1
        if (frame_count - 1) % sample_every != 0:
            continue
        
        if max_frames and results['frames_processed'] >= max_frames:
            break
        
        # Convert BGR to RGB for YOLO
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Inference
        start_time = time.time()
        detections = model.predict(frame_rgb, return_boxes=True, visualize=True)
        inference_time = (time.time() - start_time) * 1000
        
        # Extract detection results
        person_count = detections['count']
        boxes = detections.get('boxes', [])
        annotated_img = detections.get('annotated_image', None)
        
        # Store statistics
        results['person_counts'].append(person_count)
        results['frame_times'].append(inference_time)
        results['frames_processed'] += 1
        
        # Count detections by class
        for box in boxes:
            results['detections']['person'] += 1
        
        # Store sample frames
        if results['frames_processed'] % show_every == 0:
            if annotated_img is not None:
                # Convert BGR back to RGB for display
                annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
            else:
                annotated_img_rgb = frame_rgb
            
            results['sample_frames'].append({
                'frame_num': frame_count,
                'frame': annotated_img_rgb,
                'person_count': person_count,
                'inference_time': inference_time
            })
        
        # Print progress
        if results['frames_processed'] % 10 == 0:
            avg_time = np.mean(results['frame_times'][-10:])
            print(f"  Processed {results['frames_processed']} frames | "
                  f"Avg latency: {avg_time:.1f}ms | "
                  f"People detected: {person_count}")
    
    cap.release()
    return results

print("✅ Function defined: process_video_frames()")

In [ ]:
# Section 3: Define Video Path and Initialize Video Capture

# ⚠️ CONFIGURE: Change this to your video file path
# Example paths:
# - './sample_video.mp4'
# - 'C:/Users/Videos/crowd.mp4'
# - 'https://example.com/video.mp4' (URL)

VIDEO_PATH = './sample_video.mp4'  # CHANGE THIS TO YOUR VIDEO PATH

# Alternative: Use a sample video from common locations
import os
common_paths = [
    './sample_video.mp4',
    './test_video.mp4',
    '../sample_video.mp4',
    'C:/Users/Public/Videos/sample.mp4',
]

# Find first existing video
for path in common_paths:
    if os.path.exists(path):
        VIDEO_PATH = path
        print(f"✅ Found video at: {VIDEO_PATH}")
        break
else:
    print(f"⚠️ No video found in common paths")
    print(f"   Please set VIDEO_PATH to your video file")

# Initialize video capture
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"❌ Error: Could not open video at {VIDEO_PATH}")
else:
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
    
    print("✅ Video loaded successfully!")
    print(f"   Path: {VIDEO_PATH}")
    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps}")
    print(f"   Total frames: {total_frames}")
    print(f"   Duration: {duration:.2f} seconds")

In [ ]:
# Section 2: Load YOLO Model

print("🔧 Loading YOLO model...")

try:
    from models.yolo.yolov8_counter import YOLOv8Counter
    print("✅ YOLOv8Counter imported successfully")
except ImportError as e:
    print(f"❌ Error importing YOLOv8Counter: {e}")
    print("   Installing ultralytics...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    from models.yolo.yolov8_counter import YOLOv8Counter

# Initialize model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"📱 Using device: {device}")

# Load YOLOv8 model (nano for faster processing)
yolo_model = YOLOv8Counter(
    model_path='yolov8n.pt',  # nano model
    device=device,
    conf_threshold=0.5,
    iou_threshold=0.45
)

print("✅ YOLO model loaded successfully!")
print(f"   Model device: {device}")
print(f"   Confidence threshold: 0.5")
print(f"   IoU threshold: 0.45")

In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
import time
import torch
from IPython.display import display, HTML

# Add ml/src to path for imports
ml_path = Path().cwd().parent / "ml" / "src"
if str(ml_path) not in sys.path:
    sys.path.insert(0, str(ml_path))

print("✅ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# YOLO Video Testing Notebook

This notebook tests the YOLOv8 implementation for object detection and crowd counting on video files.

## Features:
- Load YOLOv8 model with GPU support
- Process video frames for real-time detection
- Detect people and other objects
- Visualize detections with bounding boxes
- Count objects by class
- Generate statistics

## Requirements:
- `ultralytics` (YOLOv8)
- `opencv-python` (cv2)
- `numpy`
- `matplotlib`
- `torch` with CUDA support (optional but recommended)